# ThinCurr Python Example: One Filament Reconstruction Create Jamfit Model

In this notebook we demonstrate how to use the jamfit.py script for creating and preparing for a filament reconstruction

## Load ThinCurr library

In [ ]:
import os 
import sys
import numpy as np
import matplotlib.pyplot as plt
thincurr_python_path = os.getenv('OFT_ROOTPATH')
if thincurr_python_path is not None:
    sys.path.append(os.path.join(thincurr_python_path,'python'))
from OpenFUSIONToolkit._core import OFT_env
from OpenFUSIONToolkit.ThinCurr import ThinCurr
from OpenFUSIONToolkit.ThinCurr.jamfit import Jamfit
from OpenFUSIONToolkit.ThinCurr.sensor import Mirnov, save_sensors, circular_flux_loop

## User Inputs and Creating a Jamfit Object

To create a Jamfit object you will need an xml file, a thincurr meshfile, and either the number of threads or an OFT_env object. The xml file can be created using the coils.py file - see the xml example on how to utilize coils.py in order to create an xml file.


In [ ]:
myOFT = OFT_env(nthreads=4)
meshfile_thincurr =  "thincurr_ex-ports.h5"
xml_name = "oft_in.xml"
jam_obj = Jamfit(xml_name, meshfile_thincurr, oft_env=myOFT)

## Defining Mirnov Sensors and Initializing Within Jamfit

In this cell we create some arbitrary sensors to measure magnetic sensor signals, as we will be generating example data to reconstruct. 

In [ ]:
sensors = []
for i, theta in enumerate(np.linspace(0.0,2.0*np.pi/20.0,12)):
    sensors.append(Mirnov(1.6*np.r_[np.cos(theta),np.sin(theta),0.0], np.r_[0.0,0.0,1.0], 'Bz_{0}'.format(i+1)))
floops_file = jam_obj.set_sensors(sensors) 
jam_obj.setup_jamfit(floops_file)

## Running time dependent synthetic simulation 
Here, we will run a time dependent synthetic simulation for two purposes:
1. To help generate a reduced model for reconstruction

2. To extract example data to be used later for reconstruction. 

In [ ]:
dt = 2.E-4
nsteps = 200
time_array = np.array(np.linspace(0, 0.03, 5)).reshape(-1,1) # must be in (num_time_steps, 1) shape 

num_coils = 1

# here we speciy coil currents (i.e shaping coils etc.) - in this example we will just have 1 coil with a constant current 
coil_currs = np.array([
    [1.E3],
    [1.E3],
    [1.E3],
    [1.E3],
    [1.E3]
])

# here we specify the plasma filament current - in this example we will only be having 1 plasma filament within the center going through a current quench 
fil_currs = np.array([
    [1.E6],
    [1.E6],
    [1.E5], 
    [1.E2],
    [1.E1]
])

all_currs = np.hstack((time_array, coil_currs, fil_currs))
hist_file = jam_obj.gen_synthetic_data(all_currs, dt, nsteps) # here we use jamfit's gen_synthetic data functio which will give us our synthetic data 

## Creating a reduced model from the time dependent synthetic run

Jamfit utilizes reduced models for reconstructions - in this cell we demonstrate one method to create a reduced model using the build-in function create_from_runTD_top_modes, which utilizes the gen_synthetic_data function from earlier to pick top 10 wall modes that are highest weighted over the time dependent simulation, in order to create a reduced model. Jamfit also allows the use of generating your own reduced model by specifiying specific eigenmodes as well - this would require using the create_reduced_model fuction on a jamfit object. 

In [ ]:
num_modes = 3
reduced_filename = "reduced_model.h5"
reduced_model, sensors_measurement, currents, eig_vecs, eig_inds, weights = jam_obj.create_from_runTD_top_modes(num_modes, reduced_filename, all_currs, dt, nsteps, verbose=True, s_freq = 1, p_freq = 1) 

## Plotting and Saving Relevant Values

Here we plot the wall mode weights over time as well as the current of plasma over time - our goal in the next notebook is to reconstruct the wall current and plasma current which we can verify by comparing the wall mode weights 

In [ ]:

temp_curr = currents['curr']
max_weights = [abs(temp_curr[:, i]).sum() for i in range(temp_curr.shape[1])] # total sum over time
print(len(max_weights))
top_modenum_indices = sorted(range(len(max_weights)), key=lambda i: max_weights[i], reverse=True)[:num_modes]
for i in top_modenum_indices:
    print(f"Mode {i}: Total weight = {max_weights[i]:.4f}")

eig_inds = np.sort(top_modenum_indices)
top_curr = temp_curr[:, eig_inds]
print(top_curr.shape)
for i in range(top_curr.shape[1]):
    plt.plot(range(top_curr.shape[0]), top_curr[:, i], marker='o', label=f'Mode {top_modenum_indices[i]}')
plt.xlabel('Time Step')
plt.ylabel('Mode Amplitude')
plt.title('Top Mode Amplitudes Over Time')
plt.legend() 
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 10))

# Plot coil currents over time
axes[0].plot(currents['time'], currents['coil_curr'][:, :num_coils])
axes[0].set_xlabel('Time')
axes[0].set_ylabel('Coil Current (A)')
axes[0].set_title('Coil Currents Over Time')
axes[0].grid(True)

# Plot plasma (filament) currents over time
axes[1].plot(currents['time'], currents['coil_curr'][:, num_coils:])
axes[1].set_xlabel('Time')
axes[1].set_ylabel('Plasma Current (A)')
axes[1].set_title('Plasma Filament Current Over Time')
axes[1].grid(True)

# Plot magnetic sensor measurements over time
axes[2].plot(sensors_measurement['time'], sensors_measurement['sensors'])
axes[2].set_xlabel('Time')
axes[2].set_ylabel('Magnetic Field (T)')
axes[2].set_title('Magnetic Sensor Measurements Over Time')
axes[2].grid(True)

plt.tight_layout()
plt.show()

## Saving values 
These arrays are being saved for the next example notebook - one_filament_reconstruct_ex.ipynb

In reality you will pipeline your own magnetic sensor data from experimental or simulation. 

In [ ]:
time_array_coilcurr = currents['time']
coil_currs = currents['coil_curr'][:, 0:num_coils]
fil_currs = currents['coil_curr'][:, num_coils:]
time_array_sensors = sensors_measurement['time']
PsiFull = sensors_measurement['sensors']

np.save('totalip.npy', fil_currs) #because there is only 1 plasma filament, the total plasma ip is the same as the filament current
np.save('time_array_coilcurr.npy', time_array_coilcurr)
np.save('coil_currs.npy', coil_currs)
np.save('time_array_sensors.npy', time_array_sensors)
np.save('sensor_measurements.npy', PsiFull)